# 🚦 YOLOv8 - Detección de Señales de Tránsito (PuzzleBot)

**Dataset:** Traffic Signs Detection (Roboflow)  
**Modelo:** YOLOv8n  
**Hardware:** Tesla T4 (Kaggle)  
**Clases:** Stop, Workers, Go straight, Turn Left, Turn Right

---

## ⚠️ ANTES DE EJECUTAR — Configuración de Kaggle

1. Click en **Settings** (esquina superior derecha)
2. **Accelerator** → `GPU T4 x2` (o `GPU P100`)
3. **Internet** → `On` (CRÍTICO, sin esto Roboflow falla)
4. **Persistence** → `Files only` (opcional, para guardar el modelo entre sesiones)
5. Click en **Save** y luego en **Run All** (▶▶)

## 📦 Celda 1 — Instalar dependencias

Instalamos `ultralytics` (YOLOv8) y `roboflow` (descarga del dataset). Esta celda **debe ejecutarse primero y completarse** antes que las siguientes.

In [ ]:
!pip install -q -U ultralytics roboflow
print("✅ Dependencias instaladas correctamente")

## 🖥️ Celda 2 — Verificar GPU y entorno

Confirmamos que la GPU está disponible y que `ultralytics` se importa sin errores.

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"PyTorch:        {torch.__version__}")
print(f"Ultralytics:    {ultralytics.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU NO detectada. Ve a Settings → Accelerator → GPU T4 x2")

# Verificación rápida del entorno de ultralytics
ultralytics.checks()

## ⬇️ Celda 3 — Descargar dataset desde Roboflow

Descarga el dataset etiquetado en formato YOLOv8.  
📍 Se guardará en `/kaggle/working/Traffic-Signs-Detection-1/`

> 💡 **Tip de seguridad:** En Kaggle es mejor guardar la API key como un *Secret*  
> (Add-ons → Secrets) y leerla con `UserSecretsClient`. Por ahora va inline para que funcione rápido.

In [ ]:
import os
from roboflow import Roboflow

# Cambiar al directorio writable de Kaggle
os.chdir("/kaggle/working")

# Conectar a Roboflow y descargar
rf = Roboflow(api_key="RcImqUfpcTHi63GIgqpb")
project = rf.workspace("gnuno").project("traffic-signs-detection-34s3q")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\n✅ Dataset descargado en: {dataset.location}")
print(f"📁 Contenido:")
!ls -la {dataset.location}

## ✂️ Celda 4 — Hacer split train/valid/test (80/10/10)

**¿Por qué esta celda?** Tu Version 1 de Roboflow no hizo el split → todas las 885 imágenes quedaron en `train/`.  
YOLOv8 necesita las carpetas `valid/` y `test/` para validar y reportar métricas.

**Lo que hace:**
- Mueve imagen + su `.txt` (label) **juntos** a `valid/` y `test/`
- Las que NO se mueven se quedan en `train/`
- Es **idempotente**: si ya hiciste el split antes, no lo rehace

**Resultado esperado** (con 885 imágenes totales):
- train: ~709, valid: ~88, test: ~88

In [ ]:
import shutil
import random
from pathlib import Path

random.seed(42)  # reproducibilidad

base = Path(dataset.location)
train_img = base / "train" / "images"
train_lbl = base / "train" / "labels"

# Solo dividir si todavía no se ha hecho
valid_dir = base / "valid" / "images"
if valid_dir.exists() and len(list(valid_dir.glob("*"))) > 0:
    print("✅ Ya existe split, saltando esta celda...")
else:
    # Listar todas las imágenes actuales en train/
    all_images = sorted(list(train_img.glob("*.jpg")) + list(train_img.glob("*.png")))
    print(f"📊 Total de imágenes en train/: {len(all_images)}")

    # Shuffle y dividir 80/10/10
    random.shuffle(all_images)
    n = len(all_images)
    n_val  = int(n * 0.10)
    n_test = int(n * 0.10)
    n_train = n - n_val - n_test

    splits = {
        "valid": all_images[:n_val],
        "test":  all_images[n_val:n_val + n_test],
        # train se queda con el resto (no se mueve)
    }

    # Crear carpetas y mover
    for split_name, files in splits.items():
        (base / split_name / "images").mkdir(parents=True, exist_ok=True)
        (base / split_name / "labels").mkdir(parents=True, exist_ok=True)

        for img_path in files:
            # Mover imagen
            shutil.move(str(img_path), str(base / split_name / "images" / img_path.name))
            # Mover su label correspondiente (.txt) — viajan juntos
            lbl_path = train_lbl / (img_path.stem + ".txt")
            if lbl_path.exists():
                shutil.move(str(lbl_path), str(base / split_name / "labels" / lbl_path.name))

    print(f"\n✅ Split realizado:")
    print(f"   train: {n_train} imágenes")
    print(f"   valid: {n_val} imágenes")
    print(f"   test:  {n_test} imágenes")

# Verificación final — imágenes y labels DEBEN coincidir en cada split
print("\n📁 Estructura final (imágenes y labels deben ser iguales):")
for split in ["train", "valid", "test"]:
    img_dir = base / split / "images"
    lbl_dir = base / split / "labels"
    n_img = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    status = "✅" if n_img == n_lbl else "⚠️"
    print(f"   {status} {split:6s}: {n_img} imágenes, {n_lbl} labels")

## 🔧 Celda 5 — Arreglar `data.yaml` (FIX importante para Kaggle)

Roboflow genera un `data.yaml` con rutas relativas que pueden fallar en Kaggle.  
Las cambiamos a rutas **absolutas** y verificamos las clases.

In [ ]:
import yaml
from pathlib import Path

dataset_path = Path(dataset.location)
yaml_path = dataset_path / "data.yaml"

# Leer el yaml original
with open(yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("📄 data.yaml ORIGINAL:")
print(yaml.dump(data_config, default_flow_style=False))

# Reescribir con rutas absolutas
data_config["path"] = str(dataset_path)
data_config["train"] = "train/images"
data_config["val"] = "valid/images"
data_config["test"] = "test/images"

with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print("\n📄 data.yaml CORREGIDO:")
print(yaml.dump(data_config, default_flow_style=False))

print(f"\n🏷️ Clases ({data_config['nc']}): {data_config['names']}")

## 🎯 Celda 6 — Entrenar YOLOv8

Empezamos con `yolov8n.pt` (nano: rápido y suficiente para 5 clases del simulador).  
Si después quieres más precisión, prueba `yolov8s.pt` (small).

**Hiperparámetros:**
- `epochs=50` — buen balance entre tiempo y aprendizaje. Si tu mAP no converge, sube a 100.
- `imgsz=640` — estándar de YOLO
- `batch=16` — cabe bien en T4 (15GB VRAM)
- `patience=15` — early stopping si no mejora en 15 épocas

⏱️ Tiempo estimado: **~30-45 min** con T4 para ~700 imágenes.

In [ ]:
from ultralytics import YOLO

# Cargar modelo preentrenado en COCO
model = YOLO("yolov8n.pt")

# Entrenar
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="/kaggle/working/runs",
    name="traffic_signs",
    exist_ok=True,
    plots=True,           # Genera gráficas de entrenamiento
    save=True,            # Guarda checkpoints
    device=0,             # GPU 0
    workers=2,            # Kaggle tiene pocos CPUs
    verbose=True,
)

print("\n✅ Entrenamiento completado")
print(f"📁 Resultados guardados en: /kaggle/working/runs/traffic_signs/")

## 📊 Celda 7 — Validar el modelo y ver métricas

Evaluamos sobre el conjunto de validación con el `best.pt` (los mejores pesos).

In [ ]:
best_weights = "/kaggle/working/runs/traffic_signs/weights/best.pt"

# Cargar el mejor modelo
best_model = YOLO(best_weights)

# Validar
metrics = best_model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=16,
    project="/kaggle/working/runs",
    name="validation",
    exist_ok=True,
)

print("\n📊 MÉTRICAS GLOBALES:")
print(f"   mAP@0.5      : {metrics.box.map50:.4f}")
print(f"   mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"   Precision    : {metrics.box.mp:.4f}")
print(f"   Recall       : {metrics.box.mr:.4f}")

print("\n📊 MÉTRICAS POR CLASE:")
class_names = data_config["names"]
for i, name in enumerate(class_names):
    print(f"   {name:15s} → mAP@0.5: {metrics.box.maps[i]:.4f}")

## 📈 Celda 8 — Visualizar curvas y matriz de confusión

In [ ]:
from IPython.display import Image, display
import os

results_dir = "/kaggle/working/runs/traffic_signs"

# Imágenes que YOLO genera automáticamente
plots = [
    "results.png",          # Curvas de loss y métricas
    "confusion_matrix.png", # Matriz de confusión
    "F1_curve.png",         # Curva F1
    "PR_curve.png",         # Precision-Recall
]

for plot in plots:
    plot_path = os.path.join(results_dir, plot)
    if os.path.exists(plot_path):
        print(f"\n📊 {plot}")
        display(Image(filename=plot_path))
    else:
        print(f"⚠️ No se encontró {plot}")

## 🔍 Celda 9 — Probar el modelo en imágenes de test

Inferencia sobre el split de **test** (que el modelo nunca vio durante entrenamiento).

In [ ]:
import glob
import random
from IPython.display import Image, display

# Tomar 6 imágenes aleatorias del set de test
test_images = glob.glob(f"{dataset_path}/test/images/*.jpg") + \
              glob.glob(f"{dataset_path}/test/images/*.png")

if not test_images:
    # Fallback: usar imágenes de validación si no hay test
    test_images = glob.glob(f"{dataset_path}/valid/images/*.jpg") + \
                  glob.glob(f"{dataset_path}/valid/images/*.png")

random.seed(42)
sample = random.sample(test_images, min(6, len(test_images)))

# Predecir
results = best_model.predict(
    source=sample,
    conf=0.25,
    iou=0.45,
    save=True,
    project="/kaggle/working/runs",
    name="predictions",
    exist_ok=True,
)

# Mostrar predicciones
pred_dir = "/kaggle/working/runs/predictions"
for img_path in glob.glob(f"{pred_dir}/*.jpg")[:6]:
    print(f"\n🖼️ {os.path.basename(img_path)}")
    display(Image(filename=img_path))

## 💾 Celda 10 — Copiar `best.pt` a la raíz para descarga fácil

Después de ejecutar esta celda:
1. En el panel derecho de Kaggle, abre **Output** → `/kaggle/working/`
2. Verás `best.pt` listo para descargar (click derecho → Download)

In [ ]:
import shutil

src = "/kaggle/working/runs/traffic_signs/weights/best.pt"
dst = "/kaggle/working/best.pt"

shutil.copy(src, dst)

size_mb = os.path.getsize(dst) / (1024 * 1024)
print(f"✅ Modelo copiado a: {dst}")
print(f"📦 Tamaño: {size_mb:.2f} MB")

# También copiamos last.pt como backup
src_last = "/kaggle/working/runs/traffic_signs/weights/last.pt"
if os.path.exists(src_last):
    shutil.copy(src_last, "/kaggle/working/last.pt")
    print(f"✅ Backup last.pt copiado también")

print("\n📁 Archivos disponibles para descargar en /kaggle/working/:")
for f in sorted(os.listdir("/kaggle/working")):
    full = os.path.join("/kaggle/working", f)
    if os.path.isfile(full):
        s = os.path.getsize(full) / (1024 * 1024)
        print(f"   📄 {f}  ({s:.2f} MB)")

## 🚀 Siguientes pasos

1. **Descarga `best.pt`** desde el panel Output de Kaggle
2. **Pruébalo localmente** con tu PuzzleBot:
   ```python
   from ultralytics import YOLO
   model = YOLO("best.pt")
   results = model.predict(source=0, show=True)  # source=0 → webcam
   ```
3. **Si el mAP es bajo** (<0.7):
   - Más imágenes / mejor etiquetado en Roboflow
   - Aumenta `epochs` a 100
   - Prueba `yolov8s.pt` (más grande, más preciso)
   - Activa augmentations en Roboflow al generar la versión
4. **Para deploy en el robot**: exporta a ONNX o TensorRT:
   ```python
   model.export(format="onnx")  # o "engine" para TensorRT
   ```